# 07 — Nos données terrain Hanoï

**Le maillon entre la collecte (ODK/Kobo) et le modèle (notebook 08).**

Ce notebook fait tout le travail sur NOS données :
1. Charge l'export CSV de KoboToolbox
2. Nettoie (même pipeline que le notebook 02 sur Sunbird)
3. Enrichit avec la météo (API Open-Meteo, gratuite)
4. Analyses de la Phase 3 du plan : time-series par site, transport vs chantier, dépassements de normes
5. Sauvegarde `data/raw/hanoi/measurements.csv` → consommé par le notebook 08

**Comment exporter depuis Kobo :** ton projet → Data → Downloads → CSV → place le fichier dans `data/raw/hanoi/`

In [ ]:
import sys, os
# feature and path code now lives in the installed package (pip install -e .)
import prepare_field_data as pfd   # source de vérité unique du nettoyage terrain

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Charge l'export Kobo le plus récent → nettoyage + backfill + météo (cf. le script).
# Le brut Kobo n'est pas modifié ; tout le travail est en mémoire.
df = pfd.build_dataframe()
print(f'{len(df)} mesures propres — sites : {df["site"].value_counts().to_dict()}')
df[['timestamp', 'site', 'collector', 'noise_dB', 'dist_to_road', 'mic_to_source']].head()

## Météo

La météo (Open-Meteo) est ajoutée automatiquement par `pfd.build_dataframe()` —
colonnes `temperature_2m`, `wind_speed_10m`, `precipitation` quand l'API répond.

## Analyse Phase 3 — 4 visualisations

Valeur de référence : **QCVN 26:2010/BTNMT**, zone ordinaire — 70 dB de 6 h à 21 h, 55 dB de 21 h à 6 h.

> Les valeurs guides OMS (53 / 45 dB) ont été **retirées** : ce sont des `L_den` / `L_night`,
> moyennes **annuelles** avec pénalités soir/nuit, non comparables à nos échantillons de 25 s.
> Et nos dépassements QCVN sont une **statistique descriptive** de l'échantillon, pas un constat
> de non-conformité : notre grandeur et nos capteurs ne sont pas ceux que la norme prescrit.
> Voir `docs/metrology.md`.


In [ ]:
# === Tâche 1a : cycle horaire par site (patterns sur une journée) ===
# Seuils QCVN 26:2010/BTNMT uniquement. Les valeurs OMS (53/45) ont ete RETIREES :
# ce sont des L_den / L_night, moyennes ANNUELLES avec penalites soir/nuit, non
# comparables a nos echantillons de 25 s (cf. docs/metrology.md).
QCVN_D, QCVN_N = 70, 55
fig, ax = plt.subplots(figsize=(11, 5))
for s in df.site.unique():
    g = df[df.site == s].groupby('hour')['noise_dB'].median()
    ax.plot(g.index, g.values, marker='o', lw=2, label=s)
ax.axhline(QCVN_D, ls='--', c='red', alpha=.7, label='QCVN jour 70')
ax.axhline(QCVN_N, ls='--', c='darkred', alpha=.7, label='QCVN nuit 55')
ax.set_xlabel('Heure'); ax.set_ylabel('$L_{A,25s}$ médian (dB)'); ax.set_xticks(range(0, 24, 2))
ax.set_title('Cycle horaire du bruit par site'); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=.3)
print('Rappel : niveaux calibres en RELATIF (contrastes), pas en absolu — smartphones non certifies.')
plt.tight_layout(); plt.savefig('../results/figures/analyse_1_horaire.png', dpi=130); plt.show()

In [ ]:
# === Tâche 1b : profil par jour de la semaine ===
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
fr = ['Lun','Mar','Mer','Jeu','Ven','Sam','Dim']
df['dayname'] = df['timestamp'].dt.day_name()
fig, ax = plt.subplots(figsize=(11, 5))
for s in df.site.unique():
    g = df[df.site == s].groupby('dayname')['noise_dB'].median().reindex(order)
    ax.plot(range(7), g.values, marker='s', lw=2, label=s)
ax.axhline(QCVN_D, ls='--', c='red', alpha=.7, label='QCVN jour 70')
ax.set_xticks(range(7)); ax.set_xticklabels(fr); ax.set_ylabel('dB médian')
ax.set_title('Profil par jour de la semaine'); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('../results/figures/analyse_2_jour.png', dpi=130); plt.show()

In [ ]:
# === Tâche 2 : transport vs construction (côte à côte) ===
df['is_constr'] = df['class'].astype(str).str.contains('construction', case=False)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5))
data = [df[~df.is_constr].noise_dB.dropna(), df[df.is_constr].noise_dB.dropna()]
a1.boxplot(data, tick_labels=['Transport/autre', 'Construction'])  # 'labels=' retire en matplotlib 3.9+
a1.axhline(QCVN_D, ls='--', c='red', alpha=.7, label='QCVN 70'); a1.set_ylabel('dB')
a1.set_title('Transport vs Construction (global)'); a1.legend(fontsize=8); a1.grid(alpha=.3)
sites = df.site.unique(); w = .35
for i, s in enumerate(sites):
    sub = df[df.site == s]
    a2.bar(i-w/2, sub[~sub.is_constr].noise_dB.median(), w, color='#1f77b4', label='Transport' if i == 0 else '')
    cv = sub[sub.is_constr].noise_dB.median()
    a2.bar(i+w/2, 0 if np.isnan(cv) else cv, w, color='#ff7f0e', label='Construction' if i == 0 else '')
a2.axhline(QCVN_D, ls='--', c='red', alpha=.7); a2.set_xticks(range(len(sites)))
a2.set_xticklabels([s.replace(' ', chr(10)) for s in sites], fontsize=8)
a2.set_ylabel('dB médian'); a2.set_title('Par site'); a2.legend(fontsize=8); a2.grid(alpha=.3)
plt.tight_layout(); plt.savefig('../results/figures/analyse_3_type.png', dpi=130); plt.show()

In [ ]:
# === Tâches 3+4 : dépassements normes, périodes de pointe, fréquence/sévérité ===
df['period'] = np.where((df.hour >= 21) | (df.hour < 6), 'night', 'day')
df['limit'] = np.where(df.period == 'night', QCVN_N, QCVN_D)
df['exceeds'] = df.noise_dB > df['limit']

# fréquence de dépassement par heure (identifie les périodes de pointe)
fig, ax = plt.subplots(figsize=(11, 4.5))
# (pandas 3 : groupby.apply n'expose plus la colonne de groupement -> on precalcule)
pe = 100 * df.assign(_ex=df.noise_dB > df['limit']).groupby('hour')['_ex'].mean()
ax.bar(pe.index, pe.values, color=['darkred' if v > 50 else 'orange' if v > 0 else 'green' for v in pe.values])
ax.set_xlabel('Heure'); ax.set_ylabel('% mesures > QCVN'); ax.set_xticks(range(0, 24, 2))
ax.set_title('Fréquence de dépassement QCVN par heure'); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('../results/figures/analyse_4_depassement.png', dpi=130); plt.show()

peak = df.groupby('hour').noise_dB.median()
print(f'Pic de bruit : {peak.idxmax()}h ({peak.max():.0f} dB) | plus calme : {peak.idxmin()}h ({peak.min():.0f} dB)')
print(f'Part des echantillons au-dessus du seuil QCVN : {100*df.exceeds.mean():.0f}%')
print('  /!\\ statistique descriptive de notre echantillon, PAS un constat de non-conformite :')
print('      notre grandeur (L_A,25s) et nos capteurs ne sont pas ceux que la norme prescrit.')

# tableau fréquence + sévérité par site x période
summ = df.groupby(['site', 'period']).apply(lambda g: pd.Series({
    'n': len(g), 'dB_median': g.noise_dB.median(),
    'pct_depassement': 100 * g.exceeds.mean(),
    'severite_moy_dB': (g.noise_dB - g['limit'])[g.exceeds].mean()})).round(1)
summ.to_csv('../results/figures/hanoi_exceedances.csv')
print('\n=== Dépassements QCVN par site x période ==='); summ

In [ ]:
# Analyse 5 — météo vs bruit (corrélation honnête)
# Piège : la météo est confondue avec l'heure (température ↔ heure r≈0.66) et avec les
# sessions (les campagnes "points calmes" sont tombées des jours de pluie). On regarde donc
# la corrélation partielle (heure contrôlée) et l'effet pluie À SESSION ÉGALE.
import numpy as np
import matplotlib.pyplot as plt

wcols = [c for c in ['temperature_2m', 'wind_speed_10m', 'precipitation'] if c in df.columns]
dw = df.dropna(subset=wcols).copy()
dw['rain'] = dw['precipitation'] > 0

print('corrélations avec noise_dB (brute / heure contrôlée) :')
for c in wcols:
    raw = dw[c].corr(dw.noise_dB)
    rh = dw.noise_dB - np.poly1d(np.polyfit(dw.hour, dw.noise_dB, 2))(dw.hour)
    rw = dw[c] - np.poly1d(np.polyfit(dw.hour, dw[c], 2))(dw.hour)
    print(f'  {c:18} brute {raw:+.2f}   partielle {rh.corr(rw):+.2f}')

# effet pluie à session (jour+site) égale
diffs = []
for _, g in dw.groupby([dw.timestamp.dt.date, 'site']):
    if g.rain.nunique() == 2 and g.rain.value_counts().min() >= 3:
        m = g.groupby('rain').noise_dB.median()
        diffs.append(m[True] - m[False])
print(f'\neffet pluie à session égale : {np.mean(diffs):+.1f} dB en médiane '
      f'(sur {len(diffs)} sessions mixtes)')
print('=> le -10 dB brut sec/pluie vient surtout des sessions calmes tombées les jours de pluie')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, c in zip(axes, wcols):
    ax.scatter(dw[c], dw.noise_dB, s=12, alpha=.4, c='#2471a3')
    ax.set_xlabel(c); ax.set_ylabel('dB'); ax.grid(alpha=.3)
    ax.set_title(f'r brut {dw[c].corr(dw.noise_dB):+.2f}')
plt.suptitle('Météo vs bruit — corrélations brutes (voir corrélations partielles ci-dessus)')
plt.tight_layout()
plt.savefig('../results/figures/analyse_5_meteo.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Sauvegarde finale : input du notebook 08 (calibration du modèle) ----
out = pfd.save_measurements(df)
print(f'{len(df)} mesures → {out}')
print('Prochaine étape : notebook 08 (transfer learning + carte de bruit)')

In [ ]:
# Carte interactive des points (source unique : scripts/build_field_map.py)
# Popup au clic = toutes les infos du point ; couches par site + chantiers.
import build_field_map as bfm
m = bfm.build_map(df)   # sauve outputs/hanoi/hanoi_field_points.html
m